# Choice OpenBB 插件真实数据验证

本 Notebook 用真实 Choice 账号验证以下链路：

`EmQuantAPI → openbb-choice Provider → OpenBB 标准接口 → DataFrame → Excel/JSON`

运行前请确认：

1. VS Code 右上角选择的是安装了 EmQuantAPI 的项目虚拟环境；
2. 已运行项目根目录的 `安装Choice_OpenBB插件.bat`；
3. 项目根目录存在 `.env`，其中填写了 `CHOICE_USERNAME` 和 `CHOICE_PASSWORD`；
4. 不要把包含账号信息的 `.env` 上传或转发。Notebook 不会显示或导出账号密码。

## 1. 检查当前 Python 和项目位置

In [1]:
import os
import sys
from pathlib import Path

print("Python 路径：", sys.executable)
print("Python 版本：", sys.version.split()[0])
print("是否虚拟环境：", sys.prefix != sys.base_prefix)
print("Notebook 当前目录：", Path.cwd().resolve())

Python 路径： d:\minicoda3\envs\dm311\python.exe
Python 版本： 3.11.14
是否虚拟环境： False
Notebook 当前目录： D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini\notebooks


In [2]:
# 如果 Notebook 不在原项目的 notebooks 文件夹，可在这里填写项目根目录。
# 示例：PROJECT_ROOT_OVERRIDE = r"D:\\qianji_openbb_mini"
PROJECT_ROOT_OVERRIDE = ""

def find_project_root(start: Path) -> Path:
    if PROJECT_ROOT_OVERRIDE.strip():
        candidate = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
        if (candidate / "extensions" / "openbb_choice").exists():
            return candidate
        raise FileNotFoundError(f"指定的项目目录不正确：{candidate}")

    for candidate in (start, *start.parents):
        if (candidate / "extensions" / "openbb_choice").exists():
            return candidate
    raise FileNotFoundError(
        "没有找到 qianji_openbb_mini 项目。请把 Notebook 放进项目的 notebooks 文件夹，"
        "或填写 PROJECT_ROOT_OVERRIDE。"
    )

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
print("识别到的项目根目录：", PROJECT_ROOT)
print(".env 是否存在：", (PROJECT_ROOT / ".env").exists())

识别到的项目根目录： D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini
.env 是否存在： True


In [5]:
import subprocess
import sys
from pathlib import Path

choice_extension = PROJECT_ROOT / "extensions" / "openbb_choice"

print("当前Python：", sys.executable)
print("Choice扩展目录：", choice_extension)
print("目录是否存在：", choice_extension.exists())

if not choice_extension.exists():
    raise FileNotFoundError(
        f"没有找到Choice扩展目录：{choice_extension}"
    )

# 安装根项目
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-e",
        str(PROJECT_ROOT) + "[openbb]",
    ],
    cwd=PROJECT_ROOT,
    check=True,
)

# 安装独立的openbb-choice扩展
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-e",
        str(choice_extension),
    ],
    cwd=PROJECT_ROOT,
    check=True,
)

# 使用同一个Python环境重新构建OpenBB
subprocess.run(
    [
        sys.executable,
        "-c",
        "import openbb; openbb.build()",
    ],
    cwd=PROJECT_ROOT,
    check=True,
)

print("安装和构建完成")
print("请立即重启Notebook内核，然后从第一格重新运行")

当前Python： d:\minicoda3\envs\dm311\python.exe
Choice扩展目录： D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini\extensions\openbb_choice
目录是否存在： True
安装和构建完成
请立即重启Notebook内核，然后从第一格重新运行


## 2. 读取配置（不显示账号密码）

In [3]:
from dotenv import load_dotenv

ENV_PATH = PROJECT_ROOT / ".env"
if not ENV_PATH.exists():
    raise FileNotFoundError(
        f"没有找到 {ENV_PATH}。请复制 .env.example 为 .env，然后填写 Choice 配置。"
    )

load_dotenv(ENV_PATH, override=True)

choice_username = os.getenv("CHOICE_USERNAME", "").strip()
choice_password = os.getenv("CHOICE_PASSWORD", "")
choice_login_mode = os.getenv("CHOICE_LOGIN_MODE", "auto").strip() or "auto"

print("Choice 用户名已配置：", bool(choice_username))
print("Choice 密码已配置：", bool(choice_password))
print("Choice 登录模式：", choice_login_mode)

if not choice_username or not choice_password:
    raise RuntimeError(".env 中缺少 CHOICE_USERNAME 或 CHOICE_PASSWORD。")

Choice 用户名已配置： True
Choice 密码已配置： True
Choice 登录模式： userInfo


## 3. 检查 EmQuantAPI 和 OpenBB Provider

In [4]:
try:
    from EmQuantAPI import c as choice_sdk
    print("EmQuantAPI：导入成功")
except Exception as exc:
    raise RuntimeError(
        "当前 Notebook 内核无法导入 EmQuantAPI。请确认 SDK 安装在上面显示的 Python 环境中，"
        "安装后重启 Notebook 内核。"
    ) from exc

from openbb import obb

providers = list(obb.coverage.providers)
print("OpenBB 是否发现 choice Provider：", "choice" in providers)

if "choice" not in providers:
    raise RuntimeError(
        "OpenBB 尚未发现 choice Provider。请运行安装Choice_OpenBB插件.bat，"
        "确认 openbb-build 成功，然后重启 VS Code/Notebook 内核。"
    )

EmQuantAPI：导入成功
OpenBB 是否发现 choice Provider： True


## 4. 设置本次验证范围

第一次建议只验证一只股票、最近约 45 天。需要其他代码或日期时，直接修改下面三个变量。

In [5]:
from datetime import date, timedelta

default_end = date.today()
default_start = default_end - timedelta(days=45)

SYMBOL = os.getenv("VALIDATION_SYMBOLS", "000001.SZ").split(",")[0].strip()
START_DATE = os.getenv("VALIDATION_START_DATE", "").strip() or default_start.isoformat()
END_DATE = os.getenv("VALIDATION_END_DATE", "").strip() or default_end.isoformat()
PERIOD = "daily"

print("证券代码：", SYMBOL)
print("开始日期：", START_DATE)
print("结束日期：", END_DATE)
print("周期：", PERIOD)

证券代码： 000001.SZ
开始日期： 2026-07-17
结束日期： 2026-08-31
周期： daily


## 5. 通过 OpenBB 标准接口调用 Choice

下面这格会实际登录并请求 Choice。默认使用 `ForceLogin=0`，不会主动踢掉其他登录终端。

In [6]:
obb.user.credentials.choice_username = choice_username
obb.user.credentials.choice_password = choice_password

try:
    result = obb.equity.price.historical(
        symbol=SYMBOL,
        start_date=START_DATE,
        end_date=END_DATE,
        period=PERIOD,
        use_cache=False,
        provider="choice",
    )
except Exception as exc:
    print("真实调用失败：", type(exc).__name__)
    print(str(exc))
    print("请只保存 ErrorCode/ErrorMsg 和当前 Python 路径，不要截图或发送账号密码。")
    raise

print("真实调用成功")
print("provider：", result.provider)
print("返回行数：", len(result.results))

[EmQuantAPI Python] [Em_Info][2026-08-31 18:30:20]:The current version is EmQuantAPI(V2.7.5.0).

[EmQuantAPI Python] [Em_Info][2026-08-31 18:30:20]:verifying your token...

[EmQuantAPI Python] [Em_Info][2026-08-31 18:30:20]:connect server...

[EmQuantAPI Python] [Em_Info][2026-08-31 18:30:22]:token login start success!

[EmQuantAPI Python] [Em_Info][2026-08-31 18:30:24]:updating ChoiceToHQ.xml from version 0 to 120

[EmQuantAPI Python] [Em_Info][2026-08-31 18:30:26]:loading ChoiceToHQ.xml...

[EmQuantAPI Python] [Em_Info][2026-08-31 18:30:28]:DownLoad D:/EMQuantAPI_Python/python3/libs/windows/bjse_code_conversion.txt success.

[EmQuantAPI Python] [Em_Info][2026-08-31 18:30:30]:percentflag(for csd/css/cses) update success.

[EmQuantAPI Python] [Em_Info][2026-08-31 18:30:35]:heartbeatthread end.

真实调用成功
provider： choice
返回行数： 32


## 6. 查看数据并形成验收证据

In [7]:
import pandas as pd
from IPython.display import display

rows = [
    item.model_dump(mode="json") if hasattr(item, "model_dump") else dict(item)
    for item in result.results
]
df = pd.DataFrame(rows)

if df.empty:
    raise RuntimeError("接口调用没有报错，但结果为空。请检查证券代码、日期范围和账号数据权限。")

display(df.head(10))
print("字段：", list(df.columns))
print("总行数：", len(df))

,date,open,high,low,close,volume,vwap,symbol,amount,prev_close,change,change_percent,source,volume_unit,amount_unit,timezone
0,2026-07-17,10.75,10.88,10.72,10.78,107549901.0,None,000001.SZ,1.163189e+09,10.77,0.01,0.000929,choice,share,CNY,Asia/Shanghai
1,2026-07-20,10.75,11.00,10.74,10.98,156730393.0,None,000001.SZ,1.713460e+09,10.78,0.20,0.018553,choice,share,CNY,Asia/Shanghai
2,2026-07-21,10.99,11.13,10.83,10.84,175511288.0,None,000001.SZ,1.925299e+09,10.98,-0.14,-0.012750,choice,share,CNY,Asia/Shanghai
3,2026-07-22,10.81,10.98,10.77,10.98,102948394.0,None,000001.SZ,1.120151e+09,10.84,0.14,0.012915,choice,share,CNY,Asia/Shanghai
4,2026-07-23,10.92,11.12,10.90,11.08,109574268.0,None,000001.SZ,1.210838e+09,10.98,0.10,0.009107,choice,share,CNY,Asia/Shanghai
5,2026-07-24,11.09,11.18,11.09,11.10,114093292.0,None,000001.SZ,1.269361e+09,11.08,0.02,0.001805,choice,share,CNY,Asia/Shanghai
6,2026-07-27,11.11,11.16,11.04,11.11,95715556.0,None,000001.SZ,1.062796e+09,11.10,0.01,0.000901,choice,share,CNY,Asia/Shanghai
7,2026-07-28,11.10,11.21,11.09,11.20,106101129.0,None,000001.SZ,1.185515e+09,11.11,0.09,0.008101,choice,share,CNY,Asia/Shanghai
8,2026-07-29,11.19,11.36,11.18,11.28,151105407.0,None,000001.SZ,1.705864e+09,11.20,0.08,0.007143,choice,share,CNY,Asia/Shanghai
9,2026-07-30,11.28,11.62,11.18,11.61,277770773.0,None,000001.SZ,3.194072e+09,11.28,0.33,0.029255,choice,share,CNY,Asia/Shanghai


字段： ['date', 'open', 'high', 'low', 'close', 'volume', 'vwap', 'symbol', 'amount', 'prev_close', 'change', 'change_percent', 'source', 'volume_unit', 'amount_unit', 'timezone']
总行数： 32


In [8]:
date_column = "date" if "date" in df.columns else None
duplicate_count = int(df.duplicated(subset=[date_column]).sum()) if date_column else None
missing_by_column = df.isna().sum().astype(int).to_dict()

evidence = {
    "provider": str(result.provider),
    "symbol": SYMBOL,
    "requested_start_date": START_DATE,
    "requested_end_date": END_DATE,
    "period": PERIOD,
    "row_count": int(len(df)),
    "first_date": str(df[date_column].min()) if date_column else None,
    "last_date": str(df[date_column].max()) if date_column else None,
    "duplicate_date_count": duplicate_count,
    "missing_by_column": missing_by_column,
    "credentials_included": False,
}

display(pd.DataFrame([evidence]).drop(columns=["missing_by_column"]))
display(pd.DataFrame({"字段": missing_by_column.keys(), "缺失数量": missing_by_column.values()}))

,provider,symbol,requested_start_date,requested_end_date,period,row_count,first_date,last_date,duplicate_date_count,credentials_included
0,choice,000001.SZ,2026-07-17,2026-08-31,daily,32,2026-07-17,2026-08-31,0,False


,字段,缺失数量
0,date,0
1,open,0
2,high,0
3,low,0
4,close,0
5,volume,0
6,vwap,32
7,symbol,0
8,amount,0
9,prev_close,0


## 7. 导出 Excel 和 JSON

导出文件不包含 Choice 账号或密码，可作为本次真实调用和数据落地的验收证据。

In [9]:
import json
from datetime import datetime

OUTPUT_DIR = PROJECT_ROOT / "validation_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
safe_symbol = SYMBOL.replace(".", "_")
excel_path = OUTPUT_DIR / f"Choice_OpenBB真实调用_{safe_symbol}_{timestamp}.xlsx"
json_path = OUTPUT_DIR / f"Choice_OpenBB真实调用_{safe_symbol}_{timestamp}.json"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="真实数据", index=False)
    pd.DataFrame([evidence]).to_excel(writer, sheet_name="验收摘要", index=False)

json_path.write_text(
    json.dumps(
        {"evidence": evidence, "results": rows},
        ensure_ascii=False,
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)

print("Excel 已导出：", excel_path)
print("JSON 已导出：", json_path)

Excel 已导出： D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini\validation_output\Choice_OpenBB真实调用_000001_SZ_20260831_183044.xlsx
JSON 已导出： D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini\validation_output\Choice_OpenBB真实调用_000001_SZ_20260831_183044.json


## 验收完成标准

当第 5 格显示 `真实调用成功`、结果表中存在真实行情，并且最后成功生成 Excel 和 JSON 时，即可证明：

- Choice 官方 SDK 可以在当前虚拟环境中运行；
- `openbb-choice` Provider 已被 OpenBB 识别；
- `provider="choice"` 可以返回真实数据；
- 数据能够导出并形成验收证据。